# 03 — Beam Search Demo and Visualization

Demonstrates `BeamSearchPromptOptimizer` on a small WSU course advising validation set.
Uses a deterministic mock LLM so the notebook runs without an API key.
Swap in `ClaudeClient` (see bottom cell) to run against the real model.

In [ ]:
import sys
sys.path.insert(0, '../../')

from src.prompts.template import PromptTemplate
from src.prompts.mutations import PromptMutator
from src.search.beam_search import BeamSearchPromptOptimizer
from src.llm.base import BaseLLMClient

## Mock LLM

Returns the correct answer only when the rendered prompt contains `"step-by-step"` (the CoT trigger).
This lets beam search demonstrate convergence without any API calls.

In [ ]:
class CoTKeywordMockLLM(BaseLLMClient):
    """Correct answer only when Chain-of-Thought trigger is present."""
    def __init__(self, trigger='step-by-step'):
        self.trigger = trigger
        self.call_count = 0

    def generate(self, prompt, temperature=0.0, max_tokens=500):
        self.call_count += 1
        return 'YES' if self.trigger in prompt else 'NO'

    def get_usage_stats(self):
        return {'calls': self.call_count}

llm = CoTKeywordMockLLM()
print('Mock LLM ready')

## Validation Set

10 WSU course advising questions. The expected answer is `'YES'` for all of them
so accuracy = fraction of responses that contain `'yes'` (case-insensitive).

In [ ]:
VALIDATION_SET = [
    {'question': 'Can a student with MATH 171 and CPTS 121 take CPTS 223?', 'answer': 'YES'},
    {'question': 'Does completing CPTS 121 satisfy the prerequisite for CPTS 223?', 'answer': 'YES'},
    {'question': 'Is MATH 171 a prerequisite for MATH 172?', 'answer': 'YES'},
    {'question': 'Can a student enroll in CPTS 360 after CPTS 223 and MATH 216?', 'answer': 'YES'},
    {'question': 'Does a CS major need MATH 315 for graduation?', 'answer': 'YES'},
    {'question': 'Is CPTS 317 required for the CS degree?', 'answer': 'YES'},
    {'question': 'Can a student take CPTS 451 after completing CPTS 360?', 'answer': 'YES'},
    {'question': 'Does CPTS 122 require CPTS 121?', 'answer': 'YES'},
    {'question': 'Is MATH 220 a requirement for the CS degree?', 'answer': 'YES'},
    {'question': 'Can a student with 60 credits apply for upper-division courses?', 'answer': 'YES'},
]

print(f'Validation set: {len(VALIDATION_SET)} questions')

## Initial Prompt Template

In [ ]:
initial_prompt = PromptTemplate(
    task_description='Answer the following WSU course advising question. Reply YES or NO.'
)

print('=== Initial Prompt (rendered) ===')
print(initial_prompt.render(VALIDATION_SET[0]['question']))
print(f'\nMutation path: {initial_prompt.mutation_path()}')

## Run Beam Search

- `beam_width=3` — keep the top 3 candidates at each iteration
- `max_iterations=5` — up to 5 rounds of expansion
- `patience=2` — stop early if accuracy does not improve for 2 consecutive rounds

In [ ]:
optimizer = BeamSearchPromptOptimizer(
    beam_width=3,
    max_iterations=5,
    llm_client=llm,
    patience=2,
)

best_prompt = optimizer.search(initial_prompt, VALIDATION_SET)

print(f'Search complete — {len(optimizer.history)} iteration(s) ran')
print(f'API calls made: {llm.call_count}')

## Convergence History

In [ ]:
print(f'{"Iter":>4}  {"Best Acc":>9}  {"Beam Accuracies":>30}  Best Path')
print('-' * 80)
for entry in optimizer.history:
    accs = ', '.join(f'{a:.2f}' for a in entry['beam_accuracies'])
    print(f"{entry['iteration']:>4}  {entry['best_accuracy']:>9.2f}  {accs:>30}  {entry['best_path']}")

## Convergence Curve

In [ ]:
import matplotlib.pyplot as plt

iterations = [e['iteration'] for e in optimizer.history]
best_accs  = [e['best_accuracy'] for e in optimizer.history]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(iterations, best_accs, marker='o', linewidth=2, label='Best beam accuracy')

# Shade the full beam spread at each iteration
for entry in optimizer.history:
    accs = entry['beam_accuracies']
    ax.fill_between(
        [entry['iteration'], entry['iteration']],
        min(accs), max(accs),
        alpha=0.15, color='steelblue',
    )

ax.set_xlabel('Iteration')
ax.set_ylabel('Accuracy')
ax.set_title('Beam Search Convergence')
ax.set_ylim(-0.05, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Best Prompt Found

In [ ]:
print(f'Mutation path : {best_prompt.mutation_path()}')
print(f'Final accuracy: {optimizer.history[-1]["best_accuracy"]:.0%}')
print()
print('=== Best Prompt (rendered on first validation question) ===')
print(best_prompt.render(VALIDATION_SET[0]['question']))

## Running with the Real Claude API

Uncomment the block below after setting `ANTHROPIC_API_KEY` in `.env`.
The `ResponseCache` means repeated runs on the same questions cost nothing.

In [ ]:
# from dotenv import load_dotenv
# load_dotenv('../../.env')
#
# from src.llm.claude_client import ClaudeClient
# from src.llm.cache import ResponseCache
#
# real_llm = ClaudeClient(cache=ResponseCache())
#
# real_optimizer = BeamSearchPromptOptimizer(
#     beam_width=3,
#     max_iterations=5,
#     llm_client=real_llm,
#     patience=2,
# )
# real_best = real_optimizer.search(initial_prompt, VALIDATION_SET)
# print('Best path:', real_best.mutation_path())